# MQTT-Discovery vs UPnP

## Load log files

### Imports

In [ ]:
import os
import gc
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from collections.abc import Callable

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["cmr10", "Computer Modern Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
    "axes.titlesize": 16
})



### Costants

In [ ]:
log_dir = "../logs"

wired_log_dir = log_dir + "/wired"
wireless_log_dir = log_dir + "/wireless"

upnp_wired_log_dir = wired_log_dir + "/upnp"
mqtt_wired_log_dir = wired_log_dir + "/mqtt"
gena_wired_log_dir = wired_log_dir + "/gena"

upnp_wireless_log_dir = wireless_log_dir + "/upnp"
upnp_mx0_wireless_log_dir = wireless_log_dir + "/upnp_mx0"
mqtt_wireless_log_dir = wireless_log_dir + "/mqtt"
gena_wireless_log_dir = wireless_log_dir + "/gena"

### Utility functions

In [ ]:
def parse_filename(filename: str) -> tuple[int, int, int]:
    match = re.search("d-[0-9]+_(u|m)-[0-9]+_(q-[0-9]+)?", filename)

    result = ()
    if match:
        match_str = match.group()
        params = match_str.removeprefix("d-").replace("u-", "").replace("m-", "").replace("q-", "").split("_")

        result
        if "m-" in match_str:
            result = int(params[0]), int(params[1]), int(params[2])
        else:
            result = int(params[0]), int(params[1]), None
            

    return result

In [ ]:
def parse_file(file: Path, filter: Callable[[str], bool]) -> list[any]:
    result = []
    with open(file, 'r') as in_file:
        for line in in_file:
            try:
                line = line.strip()

                if filter(line):
                    data = json.loads(line)
                    
                    result.append(data)
            except:
                None
    return result

In [ ]:
def load_log_file(path: str, filter: Callable[[str], bool]) -> dict:
    result = {}

    with os.scandir(path) as files:
        for f in files:
            if f.is_file() and f.name.startswith("control") and f.name.endswith(".log"):
                params = parse_filename(f.name)
                
                if params[2] == None:
                    result.setdefault(params[0], {}).setdefault(params[1], []).append(parse_file(f, filter))
                else:
                    result.setdefault(params[0], {}).setdefault(params[1], {}).setdefault(params[2], []).append(parse_file(f, filter))
    
    return result

In [ ]:
def convert_strings_to_milliseconds(str: str) -> float:
    if str.endswith("ns"):  # Nanoseconds
        return float(str[:-2])/1000000
    elif str.endswith("µs"):  # Microseconds
        return float(str[:-2])/1000
    elif str.endswith("ms"):  # Milliseconds
        return float(str[:-2])
    elif str.endswith("s"):  # Seconds
        return float(str[:-2])*1000
    else:  # Seconds
        return float(str[:-1])

## UPnP Analysis

In [ ]:
def filter_lines(line: str) -> bool:
    return re.search("(RPC Elapsed time|Found [0-9]+ devices)", line)

upnp_tests = load_log_file(upnp_mx0_wireless_log_dir, filter_lines)

In [ ]:
def filter_lines(line: str) -> bool:
    return re.search("Event elapsed time: [0-9]+(.[0-9]+)?.s", line)

gena_tests = load_log_file(gena_wireless_log_dir, filter_lines)

### Data filtering

#### Filter latency related data

In [ ]:
latency_data = {}

for device_config, controls in upnp_tests.items():
    latency_controls = {}
    for control_config, tests in controls.items():
        latency_tests = []
        for test in tests:
            latency_values = []
            for value in test:
                match = re.search("RPC Elapsed time: [0-9]+(.[0-9]+)?.s$", value["msg"])
                if match:
                    latency_values.append(convert_strings_to_milliseconds(match.group().split(":")[1].strip()))
            
            if len(latency_values) > 0:
                latency_tests.append(latency_values)
        
        lens = np.array([len(item) for item in latency_tests])
        if len(lens) > 0:
            mask = np.arange(lens.max()) < lens[:, None]
            result = np.zeros(mask.shape, dtype=float)
            result[mask] = np.concatenate(latency_tests)
            latency_controls[control_config] = result

    latency_data[device_config] = latency_controls

#### Filter robustness related data

In [ ]:
robustness_data = {}

for device_config, controls in upnp_tests.items():
    robustness_controls = {}
    for control_config, tests in controls.items():
        robustness_tests = []
        for test in tests:
            robustness_values = []
            for value in test:
                match = re.search("Found [0-9]+ devices$", value["msg"])
                if match:
                    robustness_values.append(match.group().removeprefix("Found").removesuffix("devices").strip()) #TODO make a number from it
            
            if len(robustness_values) > 0:
                robustness_tests.append(robustness_values)
        
        lens = np.array([len(item) for item in robustness_tests])
        if len(lens) > 0:
            mask = np.arange(lens.max()) < lens[:, None]
            result = np.zeros(mask.shape, dtype=int)
            result[mask] = np.concatenate(robustness_tests)
            robustness_controls[control_config] = result

    robustness_data[device_config] = robustness_controls

#### Filter GENA related data

In [ ]:
latency_data = {}

for device_config, controls in gena_tests.items():
    latency_controls = {}
    for control_config, tests in controls.items():
        latency_tests = []
        for test in tests:
            latency_values = []
            for value in test:
                match = re.search("Event elapsed time: [0-9]+(.[0-9]+)?.s$", value["msg"])
                if match:
                    latency_values.append(convert_strings_to_milliseconds(match.group().split(":")[1].strip())) #TODO make a number from it
            
            if len(latency_values) > 0:
                latency_tests.append(latency_values)
        
        lens = np.array([len(item) for item in latency_tests])
        if len(lens) > 0:
            mask = np.arange(lens.max()) < lens[:, None]
            result = np.zeros(mask.shape, dtype=float)
            result[mask] = np.concatenate(latency_tests)
            latency_controls[control_config] = result

    latency_data[device_config] = latency_controls

### Data visualization

#### Latency visualiazation

In [ ]:
values_device = []
values_control = []

means = []
q1s = []
medians = []
q3s = []
whisker_los = []
whisker_his = []
flierss = []

for device_config, controls in latency_data.items():
    for control_config, values in controls.items():
        values_device.append(device_config)
        values_control.append(control_config)
        
        data = values[values != 0]

        means.append(np.mean(data))

        q1, median, q3 = np.percentile(data, [25, 50, 75])
        q1s.append(q1)
        medians.append(median)
        q3s.append(q3)
        iqr = q3 - q1

        whisker_lo = q1-1.5*iqr
        whisker_hi = q3+1.5*iqr

        flierss.append(data[(data < whisker_lo) | (data > whisker_hi)])
        non_fliers = data[(data >= whisker_lo) & (data <= whisker_hi)]

        whisker_los.append(np.min(non_fliers))
        whisker_his.append(np.max(non_fliers))


upnp_latency_data_pd = pd.DataFrame({"device_config": values_device,
                                "control_config": values_control,
                                "mean": means,
                                "q1": q1s,
                                "median": medians,
                                "q3": q3s,
                                "whisker_lo": whisker_los,
                                "whisker_hi": whisker_his,
                                "fliers": flierss
                                }).sort_values(by=["device_config", "control_config"])

upnp_latency_data_pd

In [ ]:
#device_configs = [1,5,10,50,100]
device_configs = upnp_latency_data_pd["device_config"].unique()
device_configs.sort()

fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))
count = 0

for device_config in device_configs:
    #config = upnp_latency_data_pd[(upnp_latency_data_pd["device_config"] == device_config) & (upnp_latency_data_pd["control_config"].isin([1,5,10,50,100]))]
    config = upnp_latency_data_pd[upnp_latency_data_pd["device_config"] == device_config]
    ax.plot(config["control_config"], config["mean"], label=f"{device_config}", color=labels_colors[count])
    count += 1

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_xlim(left=0)
ax.set_xlabel("control points [units]")
ax.set_ylabel("latency [ms]")
#ax.legend(title="Deployed devices", loc='lower right')
ax.legend(title="Deployed devices", loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()

plt.show()

In [ ]:
device_configs = [1,5,10,50,100]
#device_configs = upnp_latency_data_pd["device_config"].unique()
device_configs.sort()

fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))
print(np.shape(device_configs))
x_indexes = np.arange(np.shape(device_configs)[0])
labels = ["1","5","10","50","100"]
#labels = [f"{label}" for label in upnp_latency_data_pd["device_config"].unique()]
count = 0

for device_config in device_configs:
    config = upnp_latency_data_pd[(upnp_latency_data_pd["device_config"] == device_config) & (upnp_latency_data_pd["control_config"].isin([1,5,10,50,100]))]
    #config = upnp_latency_data_pd[upnp_latency_data_pd["device_config"] == device_config]
    
    stats = []
    for _, data in config.iterrows():
        stats.append({
            "label": f"{config["control_config"]}",
            "whislo": data["whisker_lo"],
            "q1": data["q1"],
            "med": data["median"],
            "q3": data["q3"],
            "whishi": data["whisker_hi"],
            "fliers": []
        })
    
    ax.bxp(
        stats,
        positions=[x_index+(-1+count/2)*0.3 for x_index in x_indexes],
        widths=0.1,
        patch_artist=True,
        boxprops=dict(facecolor=labels_colors[count], linewidth=0.5),
        medianprops=dict(color="black"),
        whiskerprops=dict(color=labels_colors[count]),
        capprops=dict(color=labels_colors[count]),
        flierprops=dict(markerfacecolor=labels_colors[count], markeredgecolor=labels_colors[count])
        )

    count += 1

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_xticks(x_indexes)
ax.set_xticklabels(labels)
plt.xlabel("control points [units]")
plt.ylabel("latency [ms]")

legend_patches = [Patch(facecolor=labels_colors[n], label=labels[n]) for n in range(len(labels))]
ax.legend(handles=legend_patches, title="Deployed devices", loc='upper left')
plt.tight_layout()
plt.show()

#### Robustness visualization

In [ ]:
values_device = []
values_control = []

means = []
q1s = []
medians = []
q3s = []
whisker_los = []
whisker_his = []
flierss = []

for device_config, controls in robustness_data.items():
    for control_config, tests in controls.items():
        values_device.append(device_config)
        values_control.append(control_config)

        errors = np.array(1 - tests/device_config)*100

        means.append(np.mean(errors))
        
        q1, median, q3 = np.percentile(errors, [25, 50, 75])
        q1s.append(q1)
        medians.append(median)
        q3s.append(q3)
        iqr = q3 - q1

        whisker_lo = q1-1.5*iqr
        whisker_hi = q3+1.5*iqr

        flierss.append(errors[(errors < whisker_lo) | (errors > whisker_hi)])
        non_fliers = errors[(errors >= whisker_lo) & (errors <= whisker_hi)]

        whisker_los.append(np.min(non_fliers))
        whisker_his.append(np.max(non_fliers))


upnp_robustness_data_pd = pd.DataFrame({"device_config": values_device,
                                   "control_config": values_control,
                                   "mean": means,
                                   "q1": q1s,
                                   "median": medians,
                                   "q3": q3s,
                                   "whisker_lo": whisker_los,
                                   "whisker_hi": whisker_his,
                                   "fliers": flierss
                                   }).sort_values(by=["device_config", "control_config"])

upnp_robustness_data_pd

In [ ]:
device_configs = [1,5,10,50,100]
device_configs = upnp_robustness_data_pd["device_config"].unique()
#device_configs.sort()

fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))
count = 0
for device_config in device_configs:
    #robustValues = upnp_robustness_data_pd[(upnp_robustness_data_pd["device_config"] == device_config) & (upnp_robustness_data_pd["control_config"].isin([1,5,10,50,100]))]
    robustValues = upnp_robustness_data_pd[upnp_robustness_data_pd["device_config"] == device_config]

    ax.plot(robustValues["control_config"], robustValues["mean"], label=f"{device_config}", color=labels_colors[count])

    count += 1

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_xlim(left=0)
ax.set_xlabel("control points [units]")
ax.set_ylabel("error rate [%]")
#ax.legend(title="Deployed devices", loc='lower right')
ax.legend(title="Deployed devices", loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
    
plt.show()

In [ ]:
device_configs = [1,5,10,50,100]
#device_configs = upnp_latency_data_pd["device_config"].unique()
device_configs.sort()

fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))
x_indexes = np.arange(np.shape(device_configs)[0])
labels = ["1","5","10","50","100"]
#labels = [f"{label}" for label in upnp_latency_data_pd["device_config"].unique()]
count = 0

for device_config in device_configs:
    robustValues = upnp_robustness_data_pd[(upnp_robustness_data_pd["device_config"] == device_config) & (upnp_robustness_data_pd["control_config"].isin([1,5,10,50,100]))]
    #robustValues = upnp_robustness_data_pd[upnp_robustness_data_pd["device_config"] == device_config]
    
    stats = []
    for _, data in robustValues.iterrows():
        stats.append({
            "label": f"{robustValues["control_config"]}",
            "whislo": data["whisker_lo"],
            "q1": data["q1"],
            "med": data["median"],
            "q3": data["q3"],
            "whishi": data["whisker_hi"],
            "fliers": data["fliers"]
        })
    
    ax.bxp(
        stats,
        positions=[x_index+(-1+count/2)*0.3 for x_index in x_indexes],
        widths=0.1,
        patch_artist=True,
        boxprops=dict(facecolor=labels_colors[count], linewidth=0.5),
        medianprops=dict(color="black"),
        whiskerprops=dict(color=labels_colors[count]),
        capprops=dict(color=labels_colors[count]),
        flierprops=dict(markerfacecolor=labels_colors[count], markeredgecolor=labels_colors[count])

        )

    count += 1

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_xticks(x_indexes)
ax.set_xticklabels(labels)
plt.xlabel("control points [units]")
plt.ylabel("error rate [%]")

legend_patches = [Patch(facecolor=labels_colors[n], label=labels[n]) for n in range(len(labels))]
ax.legend(handles=legend_patches, title="Deployed devices", loc='center left')
plt.tight_layout()
plt.show()

## MQTT Analysis

### Data filtering

#### Filter latency related data

In [ ]:
def filter_mqtt_latency(mqtt_tests: dict) -> dict:
    latency_data = {}

    for device_config, controls in mqtt_tests.items():
        latency_controls = {}
        for control_config, control_tests in controls.items():
            latency_qos = {}
            for qos_config, qos_tests in control_tests.items():
                latency_tests = []
                for test in qos_tests:
                    latency_values = []
                    for value in test:
                        match = re.search("elapsed set: [0-9]+(.[0-9]+)?.s$", value["msg"])
                        if match:
                            latency_values.append(convert_strings_to_milliseconds(match.group().split(":")[1].strip()))
                    
                    if len(latency_values) > 0:
                        latency_tests.append(latency_values)
                
                lens = np.array([len(item) for item in latency_tests])
                if len(lens) > 0:
                    mask = np.arange(lens.max()) < lens[:, None]
                    result = np.zeros(mask.shape, dtype=float)
                    result[mask] = np.concatenate(latency_tests)
                    latency_qos[qos_config] = result
            
            latency_controls[control_config] = latency_qos

        latency_data[device_config] = latency_controls
    
    return latency_data

In [ ]:
latency_data = {}

mqtt = mqtt_wireless_log_dir

def filter_lines(line: str) -> bool:
    return re.search("elapsed set: [0-9]+(.[0-9]+)?.s", line)

for qos in range(0,3):
    for devices in range(1,10):
        mqtt_latency_part = filter_mqtt_latency(load_log_file(mqtt + f"/d-{devices}_q-{qos}", filter_lines))

        for devices_key, devices_config in mqtt_latency_part.items():
            for controls_key, controls_config in devices_config.items():
                for qos_key, value in controls_config.items():
                    latency_data.setdefault(devices_key, {}).setdefault(controls_key, {})[qos_key] = value
    
    for devices in range(10,101,10):
        mqtt_latency_part = filter_mqtt_latency(load_log_file(mqtt + f"/d-{devices}_q-{qos}", filter_lines))

        for devices_key, devices_config in mqtt_latency_part.items():
            for controls_key, controls_config in devices_config.items():
                for qos_key, value in controls_config.items():
                    latency_data.setdefault(devices_key, {}).setdefault(controls_key, {})[qos_key] = value
    
    gc.collect()

#### Filter robustness related data

In [ ]:
def filter_mqtt_robustness(mqtt_tests: dict) -> dict:
    robustness_data = {}

    for device_config, controls in mqtt_tests.items():
        robustness_controls = {}
        for control_config, control_tests in controls.items():
            robustness_tests = {}
            for qos_config, qos_tests in control_tests.items():
                robustness_qos = []
                for test in qos_tests:
                    robustness_values = []
                    for value in test:
                        match = re.search("Mqtt search found [0-9]+", value["msg"])
                        if match:
                            robustness_values.append(match.group().removeprefix("Mqtt search found "))
                    
                    if len(robustness_values) > 0:
                        robustness_qos.append(robustness_values)
                
                lens = np.array([len(item) for item in robustness_qos])
                if len(lens) > 0:
                    mask = np.arange(lens.max()) < lens[:, None]
                    result = np.zeros(mask.shape, dtype=float)
                    result[mask] = np.concatenate(robustness_qos)
                    robustness_tests[qos_config] = result
            
            robustness_controls[control_config] = robustness_tests

        robustness_data[device_config] = robustness_controls

    return robustness_data


In [ ]:
robustness_data = {}

mqtt = mqtt_wireless_log_dir

def filter_lines(line: str) -> bool:
    return re.search("Mqtt search found [0-9]+", line)

for qos in range(0,3):
    for devices in range(1,10):
        mqtt_robustness_part = filter_mqtt_robustness(load_log_file(mqtt + f"/d-{devices}_q-{qos}", filter_lines))

        for devices_key, devices_config in mqtt_robustness_part.items():
            for controls_key, controls_config in devices_config.items():
                for qos_key, value in controls_config.items():
                    robustness_data.setdefault(devices_key, {}).setdefault(controls_key, {})[qos_key] = value
    
    for devices in range(10,101,10):
        mqtt_robustness_part = filter_mqtt_robustness(load_log_file(mqtt + f"/d-{devices}_q-{qos}", filter_lines))

        for devices_key, devices_config in mqtt_robustness_part.items():
            for controls_key, controls_config in devices_config.items():
                for qos_key, value in controls_config.items():
                    robustness_data.setdefault(devices_key, {}).setdefault(controls_key, {})[qos_key] = value
    
    gc.collect()

### Data visualization

#### Latency visualiazation

In [ ]:
values_device = []
values_control = []
values_qos = []

means = []
q1s = []
medians = []
q3s = []
whisker_los = []
whisker_his = []
flierss = []

for device_config, controls in latency_data.items():
    for control_config, qos in controls.items():
        for qos_config, values in qos.items(): 
            values_device.append(device_config)
            values_control.append(control_config)
            values_qos.append(qos_config)

            means.append(np.mean(values))

            q1, median, q3 = np.percentile(values, [25, 50, 75])
            q1s.append(q1)
            medians.append(median)
            q3s.append(q3)
            iqr = q3 - q1

            whisker_lo = q1-1.5*iqr
            whisker_hi = q3+1.5*iqr

            flierss.append(values[(values < whisker_lo) | (values > whisker_hi)])
            non_fliers = values[(values >= whisker_lo) & (values <= whisker_hi)]

            whisker_los.append(np.min(non_fliers))
            whisker_his.append(np.max(non_fliers))

        
mqtt_latency_data_pd = pd.DataFrame({"device_config": values_device,
                                "control_config": values_control,
                                "qos_config": values_qos,
                                "mean": means,
                                "q1": q1s,
                                "median": medians,
                                "q3": q3s,
                                "whisker_lo": whisker_los,
                                "whisker_hi": whisker_his,
                                "fliers": flierss
                                }).sort_values(by=["device_config","control_config","qos_config"])

mqtt_latency_data_pd

In [ ]:
qos_configs = mqtt_latency_data_pd["qos_config"].unique()
qos_configs.sort()

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True, sharex=True)
labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))

for qos_config in qos_configs:
    qos_values = mqtt_latency_data_pd[mqtt_latency_data_pd["qos_config"] == qos_config]

    device_configs = qos_values["device_config"].unique()
    device_configs.sort()

    count = 0
    for device_config in device_configs:
        config = qos_values[(qos_values["device_config"] == device_config) & (qos_values["control_config"].isin([1,5,10,50,100]))]
        
        ax[qos_config].plot(config["control_config"], config["mean"], label=f"{device_config}", color=labels_colors[count%len(device_configs)])
        
        count += 1
    
    ax[qos_config].set_xlabel("control points [units]")
    ax[qos_config].set_title(f"QOS {qos_config}")
    if qos_config == 0:
        ax[qos_config].set_ylabel("latency [ms]")

    ax[qos_config].grid(
        color='gray', 
        linestyle='--', 
        linewidth=0.5, 
        alpha=0.5
    )
    ax[qos_config].set_xlim(left=0.5)

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Deployed devices", loc='upper left', bbox_to_anchor=(0.075, 0.9))
#fig.legend(handles, labels, title="Deployed devices", loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()    

plt.show()

In [ ]:
device_configs = [1,5,10,50,100]
#device_configs = mqtt_latency_data_pd["device_config"].unique()
device_configs.sort()

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True, sharex=True)

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))
x_indexes = np.arange(np.shape(device_configs)[0])
labels = ["1","5","10","50","100"]
#labels = [f"{label}" for label in upnp_latency_data_pd["device_config"].unique()]

for qos_config in qos_configs:
    config = qos_values[(qos_values["device_config"] == device_config) & (qos_values["control_config"].isin([1,5,10,50,100]))]

    device_configs = qos_values["device_config"].unique()
    device_configs.sort()
    
    count = 0

    for device_config in device_configs:
        config = qos_values[qos_values["device_config"] == device_config]
        
        stats = []
        for _, data in config.iterrows():
            stats.append({
                "label": f"{config["control_config"]}",
                "whislo": data["whisker_lo"],
                "q1": data["q1"],
                "med": data["median"],
                "q3": data["q3"],
                "whishi": data["whisker_hi"],
                "fliers": []
            })
        
        ax[qos_config].bxp(
            stats,
            positions=[x_index+(-1+count/2)*0.3 for x_index in x_indexes],
            widths=0.1,
            patch_artist=True,
            boxprops=dict(facecolor=labels_colors[count%len(device_configs)], linewidth=0.5),
            medianprops=dict(color="black"),
            whiskerprops=dict(color=labels_colors[count%len(device_configs)]),
            capprops=dict(color=labels_colors[count%len(device_configs)]),
            flierprops=dict(markerfacecolor=labels_colors[count%len(device_configs)], markeredgecolor=labels_colors[count%len(device_configs)])
            )

        count += 1

    ax[qos_config].grid(
        color='gray', 
        linestyle='--', 
        linewidth=0.5, 
        alpha=0.5
    )
    ax[qos_config].set_xticks(x_indexes)
    ax[qos_config].set_xticklabels(labels)
    ax[qos_config].set_title(f"QOS {qos_config}")
    ax[qos_config].set_xlabel("control points [units]")
    if qos_config == 0:
        ax[qos_config].set_ylabel("latency [ms]")

legend_patches = [Patch(facecolor=labels_colors[n], label=labels[n]) for n in range(len(labels))]
fig.legend(handles=legend_patches, title="Deployed devices", loc='upper center', bbox_to_anchor=(0.5, 0.9))
plt.tight_layout()
plt.show()

#### Robustness visualization

In [ ]:
values_device = []
values_control = []
values_qos = []

means = []
q1s = []
medians = []
q3s = []
whisker_los = []
whisker_his = []
flierss = []

for device_config, controls in robustness_data.items():
    for control_config, qos in controls.items():
        for qos_config, tests in qos.items(): 
            values_device.append(device_config)
            values_control.append(control_config)
            values_qos.append(qos_config)

            errors = np.array(1 - tests/(device_config*control_config))*100

            means.append(np.mean(errors))

            q1, median, q3 = np.percentile(errors, [25, 50, 75])
            q1s.append(q1)
            medians.append(median)
            q3s.append(q3)
            iqr = q3 - q1

            whisker_lo = q1-1.5*iqr
            whisker_hi = q3+1.5*iqr

            flierss.append(errors[(errors < whisker_lo) | (errors > whisker_hi)])
            non_fliers = errors[(errors >= whisker_lo) & (errors <= whisker_hi)]

            whisker_los.append(np.min(non_fliers))
            whisker_his.append(np.max(non_fliers))

        
mqtt_robustness_data_pd = pd.DataFrame({"device_config": values_device,
                                   "control_config": values_control,
                                   "qos_config": values_qos,
                                   "mean": means,
                                   "q1": q1s,
                                   "median": medians,
                                   "q3": q3s,
                                   "whisker_lo": whisker_los,
                                   "whisker_hi": whisker_his,
                                   "fliers": flierss
                                   }).sort_values(by=["device_config","control_config","qos_config"])

mqtt_robustness_data_pd

In [ ]:
qos_configs = mqtt_robustness_data_pd["qos_config"].unique()
qos_configs.sort()

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True, sharex=True)
labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))

for qos_config in qos_configs:
    qos_values = mqtt_robustness_data_pd[mqtt_robustness_data_pd["qos_config"] == qos_config]
    
    device_configs = mqtt_robustness_data_pd["device_config"].unique()
    device_configs.sort()

    count = 0
    for device_config in device_configs:
        robustValues = qos_values[qos_values["device_config"] == device_config]

        ax[qos_config].plot(robustValues["control_config"], robustValues["mean"], label=f"{device_config}", color=labels_colors[count%len(device_configs)])
        
        count += 1
    
    ax[qos_config].set_xlabel("control points [units]")
    ax[qos_config].set_title(f"QOS {qos_config}")
    if qos_config == 0:
        ax[qos_config].set_ylabel("error rate [%]")

    ax[qos_config].grid(
        color='gray', 
        linestyle='--', 
        linewidth=0.5, 
        alpha=0.5
    )
    ax[qos_config].set_xlim(left=0.5)

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Deployed devices", loc='upper left', bbox_to_anchor=(0.375, 0.9))
#fig.legend(handles, labels, title="Deployed devices", loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()    

plt.show()

In [ ]:
qos_configs = mqtt_robustness_data_pd["qos_config"].unique()
qos_configs.sort()

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True, sharex=True)
labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))

labels = ["1","5","10","50","100"]
#labels = [f"{label}" for label in upnp_latency_data_pd["device_config"].unique()]

for qos_config in qos_configs:
    qos_values = mqtt_robustness_data_pd[mqtt_robustness_data_pd["qos_config"] == qos_config]
    
    device_configs = [1,5,10,50,100]
    #device_configs = mqtt_robustness_data_pd["device_config"].unique()
    device_configs.sort()

    x_indexes = np.arange(np.shape(device_configs)[0])

    count = 0
    for device_config in device_configs:
        robustValues = qos_values[(qos_values["device_config"] == device_config) & (qos_values["control_config"].isin([1,5,10,50,100]))]
        #robustValues = qos_values[qos_values["device_config"] == device_config]

        stats = []
        for _, data in robustValues.iterrows():
            stats.append({
                "label": f"{robustValues["control_config"]}",
                "whislo": data["whisker_lo"],
                "q1": data["q1"],
                "med": data["median"],
                "q3": data["q3"],
                "whishi": data["whisker_hi"],
                "fliers": []#data["fliers"]
            })
        
        ax[qos_config].bxp(
            stats,
            positions=[x_index+(-1+count/2)*0.3 for x_index in x_indexes],
            widths=0.1,
            patch_artist=True,
            boxprops=dict(facecolor=labels_colors[count%len(device_configs)], linewidth=0.5),
            medianprops=dict(color="black"),
            whiskerprops=dict(color=labels_colors[count%len(device_configs)]),
            capprops=dict(color=labels_colors[count%len(device_configs)]),
            flierprops=dict(markerfacecolor=labels_colors[count%len(device_configs)], markeredgecolor=labels_colors[count%len(device_configs)])
            )

        count += 1
    
    ax[qos_config].set_xlabel("control points [units]")
    ax[qos_config].set_title(f"QOS {qos_config}")
    if qos_config == 0:
        ax[qos_config].set_ylabel("error rate [%]")

    ax[qos_config].grid(
        color='gray', 
        linestyle='--', 
        linewidth=0.5, 
        alpha=0.5
    )
    

    ax[qos_config].set_xticks(x_indexes)
    ax[qos_config].set_xticklabels(["1","5","10","50","100"])
    #ax[qos_config].set_xticklabels(upnp_latency_data_pd["control_config"].map(lambda x: f"{x}").unique())

legend_patches = [Patch(facecolor=labels_colors[n], label=labels[n]) for n in range(len(labels))]
fig.legend(handles=legend_patches, title="Deployed devices", loc='upper center', bbox_to_anchor=(0.5, 0.9))
plt.tight_layout()    

plt.show()

## Comparison

In [ ]:
mqtt_latency_data_pd_avg = mqtt_latency_data_pd.groupby(["device_config", "control_config"]).mean().reset_index()
w_mqtt_latency_data_pd_avg = w_mqtt_latency_data_pd.groupby(["device_config", "control_config"]).mean().reset_index()

In [ ]:
mqtt_robustness_data_pd_avg = mqtt_robustness_data_pd.groupby(["device_config", "control_config"]).mean().reset_index()
w_mqtt_robustness_data_pd_avg = w_mqtt_robustness_data_pd.groupby(["device_config", "control_config"]).mean().reset_index()

In [ ]:
mqtt_robustness_data_pd_avg

In [ ]:
fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))

config = upnp_latency_data_pd[(upnp_latency_data_pd["device_config"] == 100) & (upnp_latency_data_pd["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["average_value"], label="Wireless UPnP 100", color=labels_colors[0])

config = mqtt_latency_data_pd_avg[(mqtt_latency_data_pd_avg["device_config"] == 100) & (mqtt_latency_data_pd_avg["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["average_value"], label="Wireless MQTT-D 100", color=labels_colors[2])

config = w_upnp_latency_data_pd[(w_upnp_latency_data_pd["device_config"] == 100) & (w_upnp_latency_data_pd["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["average_value"], label="Wired UPnP 100", color=labels_colors[1])

config = w_mqtt_latency_data_pd_avg[(w_mqtt_latency_data_pd_avg["device_config"] == 100) & (w_mqtt_latency_data_pd_avg["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["average_value"], label="Wired MQTT-D 100", color=labels_colors[3])

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_yscale("log")
ax.set_xlim(left=0)
ax.set_xlabel("control points [units]")
ax.set_ylabel("latency [ms]")
#ax.set_title("Latency")
ax.set_title("Latency Wired vs Wireless")
#ax.legend(title="Deployed devices", loc='center right')
ax.legend(title="Deployed devices", loc='center right', bbox_to_anchor=(1, 0.7))
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots()

labels_colors = plt.cm.tab20(np.linspace(0, 1, 20))

config = upnp_robustness_data_pd[(upnp_robustness_data_pd["device_config"] == 100) & (upnp_robustness_data_pd["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["fails_avg_zero"]*100, label="Wireless UPnP 100", color=labels_colors[0])

config = mqtt_robustness_data_pd_avg[(mqtt_latency_data_pd_avg["device_config"] == 100) & (mqtt_latency_data_pd_avg["control_config"].isin([1,5,10,50,100]))]
ax.plot(config["control_config"], config["fails_avg_zero"]*100, label="Wireless MQTT-D 100", color=labels_colors[2])

#config = w_upnp_robustness_data_pd[(w_upnp_robustness_data_pd["device_config"] == 100) & (w_upnp_robustness_data_pd["control_config"].isin([1,5,10,50,100]))]
#ax.plot(config["control_config"], config["fails_avg_zero"]*100, label="Wired UPnP 100", color=labels_colors[1])

#config = w_mqtt_robustness_data_pd_avg[(w_mqtt_robustness_data_pd_avg["device_config"] == 100) & (w_mqtt_latency_data_pd_avg["control_config"].isin([1,5,10,50,100]))]
#ax.plot(config["control_config"], config["fails_avg_zero"]*100, label="Wired MQTT-D 100", color=labels_colors[3])

plt.grid(
    color='gray', 
    linestyle='--', 
    linewidth=0.5, 
    alpha=0.5
)
ax.set_xlim(left=0)
ax.set_xlabel("control points [units]")
ax.set_ylabel("error rate [%]")
ax.set_title("Robustness")
#ax.set_title("Robustness Wired vs Wireless")
ax.legend(title="Deployed devices", loc='center right', bbox_to_anchor=(1, 0.1))
#ax.legend(title="Deployed devices", loc='center right', bbox_to_anchor=(1, 0.15))
plt.tight_layout()